JAI SHRI GANESHA!!

In [1]:
!pip install -q numpy pandas scipy scikit-learn pyreadstat
!pip install -q transformers datasets accelerate
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
import os
import numpy as np
import pandas as pd
import pyreadstat
import torch
import torch.nn as nn
import torch.nn.functional as F

from transformers import AutoTokenizer, AutoModel
from collections import defaultdict
from tqdm import tqdm

In [4]:
# =============================================================================
# CONFIGURATION
# =============================================================================

FILE_PATH  = "/content/drive/MyDrive/TEXT_FE_DATASETS_ANN/Sales Sample(15032026).dta"

TEXT_COL   = "TEXT"
LABEL_COL  = "SAL_ACTUAL_ibes"
ID_COL     = "fiscal_year"

MODEL_NAME = "roberta-large"
SPLIT_YEAR = 2013

MAX_LENGTH = 512
STRIDE     = 128

BEST_LR     = 3e-5
BEST_EPOCHS = 30

# 09 predicted outcomes
QUANTILES = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]

SAVE_DIR = "/content/drive/MyDrive/SAL_quantile(15032026)"


TRAIN_EMB_PATH   = f"{SAVE_DIR}/SAL_qr_train_emb.pt"
TEST_EMB_PATH    = f"{SAVE_DIR}/SAL_qr_test_emb.pt"
NORM_STATS_PATH  = f"{SAVE_DIR}/SAL_qr_norm_stats.pt"
TRAIN_SEQ_PATH   = f"{SAVE_DIR}/SAL_qr_train_seq.pt"
TEST_SEQ_PATH    = f"{SAVE_DIR}/SAL_qr_test_seq.pt"

FINAL_SAVE_PATH  = f"{SAVE_DIR}/SAL_quantile_predictions_v2.dta"

REGENERATE_EMBEDDINGS = True
REBUILD_SEQUENCES      = True
RETRAIN_MODEL          = True

device = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs(SAVE_DIR, exist_ok=True)

print("Device:",device)

Device: cuda


In [5]:
# =============================================================================
# LOAD DATA BEFORE SPLIT
# =============================================================================

df,_ = pyreadstat.read_dta(FILE_PATH)

df = df[df[TEXT_COL].notna()]
df = df[df[TEXT_COL].astype(str).str.strip()!=""]

df[LABEL_COL] = pd.to_numeric(df[LABEL_COL],errors="coerce")
df = df[df[LABEL_COL].notna()]
df = df[df[LABEL_COL] > 0].copy()

# winsorize extreme values BEFORE split
#p1  = df[LABEL_COL].quantile(0.01)
#p99 = df[LABEL_COL].quantile(0.99)
#df[LABEL_COL] = df[LABEL_COL].clip(lower=p1, upper=p99)

df["label_log"] = np.log(df[LABEL_COL])
df = df.sort_values(ID_COL).reset_index(drop=True)
df["original_row_id"] = df.index

print("Rows:",len(df))

Rows: 6535


In [6]:
# =============================================================================
# SPLIT + NORMALIZATION
# =============================================================================

train_df = df[df[ID_COL] <= SPLIT_YEAR].copy()
test_df  = df[df[ID_COL] >  SPLIT_YEAR].copy()

label_mean = train_df["label_log"].mean()
label_std  = train_df["label_log"].std()
label_std  = label_std if label_std > 1e-8 else 1e-8

train_df["label_norm"] = (train_df["label_log"] - label_mean)/label_std
test_df["label_norm"]  = (test_df["label_log"]  - label_mean)/label_std

torch.save({"label_mean":label_mean,"label_std":label_std},NORM_STATS_PATH)

In [7]:
# =============================================================================
# TOKENIZER
# =============================================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [8]:
# =============================================================================
# EMBEDDINGS (REUSE OLD FILES IF PRESENT)
# =============================================================================

def generate_embeddings(split_df,name):

    roberta = AutoModel.from_pretrained(MODEL_NAME).to(device)
    roberta.eval()

    reps,ids,labels,labels_orig = [],[],[],[]

    for _,row in tqdm(split_df.iterrows(),total=len(split_df),desc=name):

        text = str(row[TEXT_COL])
        label = float(row["label_norm"])
        label_og = float(row[LABEL_COL])
        rid = int(row["original_row_id"])

        enc = tokenizer(
            text,
            max_length=MAX_LENGTH,
            truncation=True,
            stride=STRIDE,
            return_overflowing_tokens=True,
            padding="max_length",
            return_tensors="pt"
        )

        with torch.no_grad():
            for i in range(enc["input_ids"].shape[0]):

                out = roberta(
                    input_ids=enc["input_ids"][i].unsqueeze(0).to(device),
                    attention_mask=enc["attention_mask"][i].unsqueeze(0).to(device)
                )

                cls = out.last_hidden_state[:,0,:].squeeze(0).cpu()

                reps.append(cls)
                ids.append(rid)
                labels.append(label)
                labels_orig.append(label_og)

    return {"reps":reps,"ids":ids,"labels":labels,"labels_orig":labels_orig}

In [ ]:
if REGENERATE_EMBEDDINGS:
    train_cache = generate_embeddings(train_df,"train")
    test_cache  = generate_embeddings(test_df,"test")

    torch.save(train_cache,TRAIN_EMB_PATH)
    torch.save(test_cache,TEST_EMB_PATH)
else:
    train_cache = torch.load(TRAIN_EMB_PATH, weights_only=False)
    test_cache  = torch.load(TEST_EMB_PATH,  weights_only=False)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
train:   1%|▏         | 43/3386 [00:18<22:59,  2.42it/s]

In [ ]:
# =============================================================================
# BUILD DOCUMENT SEQUENCES (REUSE IF EXISTING)
# =============================================================================

def build_sequences(reps,ids,labels,labels_orig):

    seqs=defaultdict(list)
    lab={}
    lab_orig={}

    for r,i,y,yo in zip(reps,ids,labels,labels_orig):
        seqs[i].append(r)
        lab[i]=y
        lab_orig[i]=yo

    X=[torch.stack(v) for v in seqs.values()]
    y=torch.tensor([lab[i] for i in seqs.keys()])
    y_orig=torch.tensor([lab_orig[i] for i in seqs.keys()])
    rid=list(seqs.keys())

    return X,y,y_orig,rid

In [ ]:
if REBUILD_SEQUENCES:
    X_train,y_train,y_train_orig,rid_train = build_sequences(**train_cache)
    X_test,y_test,y_test_orig,rid_test     = build_sequences(**test_cache)

    torch.save({"X":X_train,"y":y_train,"y_orig":y_train_orig,"rid":rid_train},TRAIN_SEQ_PATH)
    torch.save({"X":X_test ,"y":y_test ,"y_orig":y_test_orig ,"rid":rid_test },TEST_SEQ_PATH)
else:
    tr = torch.load(TRAIN_SEQ_PATH, weights_only=False)
    te = torch.load(TEST_SEQ_PATH,  weights_only=False)

    X_train,y_train,y_train_orig,rid_train = tr["X"],tr["y"],tr["y_orig"],tr["rid"]
    X_test ,y_test ,y_test_orig ,rid_test  = te["X"],te["y"],te["y_orig"],te["rid"]

In [ ]:
# =============================================================================
#  MULTI-QUANTILE MODEL (ORDERED OUTPUTS)
# =============================================================================

class MultiQuantileModel(nn.Module):

    def __init__(self,dim,nq):
        super().__init__()

        self.lstm = nn.LSTM(
            dim,dim//2,
            num_layers=2,
            bidirectional=True,
            batch_first=True,
            dropout=0.1
        )

        self.attn = nn.Sequential(
            nn.Linear(dim,dim//4),
            nn.Tanh(),
            nn.Linear(dim//4,1)
        )

        self.base  = nn.Linear(dim,1)
        self.delta = nn.Linear(dim,nq-1)

    def forward(self,x):

        out,_ = self.lstm(x)
        w = torch.softmax(self.attn(out),dim=1)
        pooled = (out*w).sum(dim=1)

        q1 = self.base(pooled)
        d  = F.softplus(self.delta(pooled))

        qs = torch.cat([q1, q1 + torch.cumsum(d,dim=1)],dim=1)

        return qs,w.squeeze(-1)

In [ ]:
class MultiPinballLoss(nn.Module):

    def __init__(self, quantiles):
        super().__init__()
        self.q = torch.tensor(quantiles).view(1,-1)

    def forward(self, preds, target):

        q = self.q.to(preds.device)
        y = target.view(-1,1).expand_as(preds)
        e = y - preds

        loss = torch.maximum(q*e,(q-1)*e)
        return loss.mean()

In [ ]:
# =============================================================================
#  TRAIN MODEL (ONLY PART THAT MUST RUN AGAIN)
# =============================================================================

model = MultiQuantileModel(
    dim=X_train[0].shape[1],
    nq=len(QUANTILES)
).to(device)

nn.init.constant_(model.delta.bias, 0.5)   # ← ADD THIS LINE

opt = torch.optim.AdamW(model.parameters(),lr=BEST_LR,weight_decay=0.01)
loss_fn = MultiPinballLoss(QUANTILES)

for epoch in range(BEST_EPOCHS):

    model.train()
    total_loss=0

    for x,y in zip(X_train,y_train):

        x=x.unsqueeze(0).to(device)
        y=torch.tensor([y.item()],device=device)

        preds,_ = model(x)
        loss = loss_fn(preds,y)

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
        opt.step()

        total_loss+=loss.item()

    print(f"Epoch {epoch+1:>3}  loss  {total_loss/len(X_train):.6f}")

In [ ]:
TEMP = 3.5

In [ ]:
# =============================================================================
# >>>>> INFERENCE (9 OUTCOMES)
# =============================================================================

norm_stats = torch.load(NORM_STATS_PATH, weights_only=False)
label_mean = norm_stats["label_mean"]
label_std  = norm_stats["label_std"]
label_std  = label_std if label_std > 1e-8 else 1e-8   # guard against zero std

TEMP = 3.5   # upper-tail stretch factor

# ──────────────────────────────────────────────────────────────

def scale_quantile_predictions(preds_actual):
    """Stretch quantile predictions symmetrically around the median."""
    median       = preds_actual[4]   # Q50 is index 4
    preds_scaled = preds_actual.copy()
    for i, val in enumerate(preds_actual):
        if val < median:
            preds_scaled[i] = median + 1.5  * (val - median)   # lower
        else:
            preds_scaled[i] = median + TEMP * (val - median)   # upper
    return preds_scaled


def predict_quantiles(model, x_doc):

    with torch.no_grad():
        preds_norm, _ = model(x_doc.unsqueeze(0).to(device))
        preds_norm    = preds_norm.squeeze(0).cpu().numpy()
        preds_actual  = np.exp(preds_norm * label_std + label_mean)

    preds_scaled = scale_quantile_predictions(preds_actual)

    return {
        f"q{int(q*100):03d}_pred": v
        for q, v in zip(QUANTILES, preds_scaled)
    }


all_results = []

for rid, x_doc in tqdm(zip(rid_test, X_test), total=len(X_test)):
    preds = predict_quantiles(model, x_doc)
    preds["original_row_id"] = rid
    all_results.append(preds)

In [ ]:
pred_df=pd.DataFrame(all_results)

pred_df["q_median"]=pred_df["q050_pred"]
pred_df["q_interval"]=pred_df["q090_pred"]-pred_df["q010_pred"]
pred_df["q_rel_interval"]=pred_df["q_interval"]/pred_df["q050_pred"]

test_meta = df[df[ID_COL]>SPLIT_YEAR][
    ["original_row_id",ID_COL,LABEL_COL]
].copy()

pred_df = pred_df.merge(test_meta,on="original_row_id",how="left")

df_final = df.merge(
    pred_df.drop(columns=[ID_COL,LABEL_COL]),
    on="original_row_id",
    how="left"
)

df_final = df_final.drop(columns=["label_log"],errors="ignore")

for col in df_final.columns:
    if df_final[col].dtype=="object":
        df_final[col]=df_final[col].astype(str)

df_final.to_stata(FINAL_SAVE_PATH,write_index=False,version=118)

print("Saved:",FINAL_SAVE_PATH)

In [ ]:


print("\n" + "="*60)
print("SECTION EXTRA: Chunk-Level Quantile Predictions")
print("="*60)



all_chunk_reps  = test_cache["reps"]
all_chunk_ids   = test_cache["ids"]

chunk_rows = []

model.eval()

with torch.no_grad():

    for emb, rid in tqdm(zip(all_chunk_reps, all_chunk_ids),
                         total=len(all_chunk_reps),
                         desc="Chunk Quantile Inference"):

        # emb is CLS embedding [dim]
        # model expects sequence [1,T,dim] -> here T=1
        x = emb.unsqueeze(0).unsqueeze(0).to(device)

        preds_norm,_ = model(x)
        preds_norm   = preds_norm.squeeze(0).cpu().numpy()

        preds_actual = np.exp(preds_norm*label_std + label_mean)
        median_chunk = preds_actual[4]
        preds_scaled = preds_actual.copy()
        for i, val in enumerate(preds_actual):
            if val < median_chunk:
                # Lower quantiles — stretch down gently
                preds_scaled[i] = median_chunk + 1.5 * (val - median_chunk)
            else:
                # Upper quantiles — stretch up more aggressively
                preds_scaled[i] = median_chunk + 3.5 * (val - median_chunk)

        preds_actual = preds_scaled

        row = {
            "original_row_id": int(rid)
        }

        # save each quantile prediction
        for q,val in zip(QUANTILES, preds_actual):
            row[f"chunk_q{int(q*100):03d}"] = float(val)

        chunk_rows.append(row)

chunk_df = pd.DataFrame(chunk_rows)

print("Chunk-level rows:", len(chunk_df))



# For each firm-year, collect chunk predictions into lists
agg_dict = {"original_row_id": []}

for q in QUANTILES:
    agg_dict[f"chunk_q{int(q*100):03d}"] = []

grouped = chunk_df.groupby("original_row_id")

for rid, g in grouped:

    agg_dict["original_row_id"].append(rid)

    for q in QUANTILES:
        col = f"chunk_q{int(q*100):03d}"

        vals = g[col].tolist()

        # if only one chunk, still store as list
        agg_dict[col].append(vals)

chunk_agg_df = pd.DataFrame(agg_dict)

print("Documents with chunk arrays:", len(chunk_agg_df))


df_saved, _ = pyreadstat.read_dta(FINAL_SAVE_PATH)

df_saved = df_saved.merge(
    chunk_agg_df,
    on="original_row_id",
    how="left"
)

print("Merged shape:", df_saved.shape)




# Stata cannot store Python lists directly.
# We serialize arrays as pipe-separated strings.

for q in QUANTILES:
    col = f"chunk_q{int(q*100):03d}"

    df_saved[col] = df_saved[col].apply(
        lambda x: "|".join([f"{v:.6f}" for v in x]) if isinstance(x,list) else ""
    )




df_saved.to_stata(FINAL_SAVE_PATH, write_index=False, version=118)

print("\nUpdated DTA saved with chunk-level quantile arrays:")
print(FINAL_SAVE_PATH)

In [ ]:
import numpy as np
import pandas as pd
import pyreadstat
import torch
from tqdm import tqdm

TOP_K = 6  # number of highest-attention chunks to keep per document

ATTN_DTA_PATH = f"{SAVE_DIR}/SAL_attended_text.dta"

In [ ]:
rid_to_text = dict(zip(
    test_df["original_row_id"].astype(int).tolist(),
    test_df[TEXT_COL].astype(str).tolist()
))
rid_to_fyear = dict(zip(
    test_df["original_row_id"].astype(int).tolist(),
    test_df[ID_COL].tolist()
))
print(f"Test documents mapped: {len(rid_to_text):,}")

In [ ]:
def get_chunk_texts(text):
    enc = tokenizer(
        text,
        max_length                = MAX_LENGTH,
        truncation                = True,
        stride                    = STRIDE,
        return_overflowing_tokens = True,
        padding                   = "max_length",
        return_tensors            = "pt"
    )
    return [
        tokenizer.decode(
            enc["input_ids"][i],
            skip_special_tokens          = True,
            clean_up_tokenization_spaces = True
        ).strip()
        for i in range(enc["input_ids"].shape[0])
    ]

In [ ]:
rows = []
model.eval()

for rid, x_doc in tqdm(zip(rid_test, X_test),
                        total = len(rid_test),
                        desc  = "Extracting attended text"):

    rid = int(rid)

    with torch.no_grad():
        _, w = model(x_doc.unsqueeze(0).to(device))
        w = w.squeeze(0).cpu().numpy()

    n_chunks    = len(w)
    chunk_texts = get_chunk_texts(rid_to_text.get(rid, ""))

    if len(chunk_texts) > n_chunks:
        chunk_texts = chunk_texts[:n_chunks]
    elif len(chunk_texts) < n_chunks:
        chunk_texts += [""] * (n_chunks - len(chunk_texts))

    k       = min(TOP_K, n_chunks)
    top_idx = sorted(np.argsort(w)[::-1][:k])
    attended_text = " ".join(chunk_texts[i] for i in top_idx)

    rows.append({
        "original_row_id" : rid,
        ID_COL            : rid_to_fyear.get(rid),
        "attended_text"   : attended_text,
        "n_chunks"        : n_chunks
    })

attn_df = pd.DataFrame(rows)
print(f"Extraction complete: {len(attn_df):,} firm-years")

In [ ]:
attn_df_stata = attn_df.copy()
for col in attn_df_stata.select_dtypes("object").columns:
    attn_df_stata[col] = attn_df_stata[col].astype(str)

attn_df_stata.to_stata(ATTN_DTA_PATH, write_index=False, version=118)
print(f"Standalone file saved: {ATTN_DTA_PATH}")

In [ ]:
print(f"Loading main data file: {FINAL_SAVE_PATH}")
main_df, meta = pyreadstat.read_dta(FINAL_SAVE_PATH)
print(f"  Shape before merge: {main_df.shape}")

main_df = main_df.drop(
    columns=[c for c in ["attended_text", "n_chunks"] if c in main_df.columns]
)

main_df = main_df.merge(
    attn_df[["original_row_id", "attended_text", "n_chunks"]],
    on  = "original_row_id",
    how = "left"
)

main_df["attended_text"] = main_df["attended_text"].fillna("")
main_df["n_chunks"]      = main_df["n_chunks"].fillna(0).astype(int)

print(f"  Shape after merge: {main_df.shape}")
print(f"  Rows with attended_text: {(main_df['attended_text'].str.len() > 0).sum():,}")

for col in main_df.select_dtypes("object").columns:
    main_df[col] = main_df[col].astype(str)

main_df.to_stata(FINAL_SAVE_PATH, write_index=False, version=118)
print(f"Main file updated: {FINAL_SAVE_PATH}")
print(f"New columns added: attended_text, n_chunks")

In [ ]:
print("\n" + "="*60)
print("SANITY CHECK")
print("="*60)

sample = main_df[main_df["attended_text"].str.len() > 0].head(2)

for _, row in sample.iterrows():
    print(f"\n  original_row_id : {int(row['original_row_id'])}")
    print(f"  fiscal_year     : {row[ID_COL]}")
    print(f"  n_chunks        : {int(row['n_chunks'])}")
    print(f"  attended_text preview:")
    print(f"    {str(row['attended_text'])[:300]} ...")

attended_lens = main_df.loc[main_df["attended_text"].str.len() > 0,
                             "attended_text"].str.len()
full_lens     = test_df[TEXT_COL].str.len()

print(f"\n  Avg attended_text length : {attended_lens.mean():,.0f} chars")
print(f"  Avg full TEXT length     : {full_lens.mean():,.0f} chars")
print(f"  Compression ratio        : {attended_lens.mean()/full_lens.mean():.1%}")